# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [5]:
from openai import OpenAI
import gradio as gr
import json
import os
from dotenv import load_dotenv

In [10]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [11]:
system_content = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [12]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_content}] + history + [{"role" : "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn = chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [14]:
ticket_prices = {"delhi" : "₹1999", "pune": "₹1500", "kolkata": "₹5999", "leh": "₹6500"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown City")
    return f"The price of a ticket to {destination_city} is {price}"

In [17]:
get_ticket_price("delhi")

Tool called for city delhi


'The price of a ticket to delhi is ₹1999'

In [18]:
price_function = {
    "name" : "get_ticket_price",
    "description" : "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type" : "object",
        "properties" : {
            "destination_city" : {
                "type" : "string",
                "description" : "The city that the customer wants to travel to",
            }
        },
        "required" : ["destination_city"],
        "additionalProperties" : False
    }
}

In [19]:
tools = [{"type": "function", "function": price_function}]

In [36]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_content}] + history + [{"role" : "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)

        for msg in messages:
            print(msg)

        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content

In [24]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == 'get_ticket_price':
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)

        response = {
            "role" : "tool",
            "content" : price_details,
            "tool_call_id": tool_call.id
        }

    return response

In [25]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tool called for city Delhi
Tool called for city Delhi


Traceback (most recent call last):
  File "/Users/jaymomaya/Learning/AI/LLM Engineering/learning-llm/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jaymomaya/Learning/AI/LLM Engineering/learning-llm/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jaymomaya/Learning/AI/LLM Engineering/learning-llm/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jaymomaya/Learning/AI/LLM Engineering/learning-llm/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1696, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^


In [30]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [37]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Tool called for city Pune
Tool called for city Delhi
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'Hi there, I want to check flight prices for Pune and Delhi'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_atbKtTZM1WwuKpcJZhVugGlT', function=Function(arguments='{"destination_city": "Pune"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_1ByTB48uoCHXUAymTK4H4kbZ', function=Function(arguments='{"destination_city": "Delhi"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'The price of a ticket to Pune is ₹1500', 'tool_call_id': 'call_atbKtTZM1WwuKpcJZhVugGlT'}
{'role': 'tool', 'content': 'The price 

In [38]:
import sqlite3

In [39]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

Tool called for city Nepal
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'Hi there, I want to check flight prices for Pune and Delhi'}
{'role': 'assistant', 'content': 'The price of a ticket to Pune is ₹1500, and to Delhi is ₹1999.'}
{'role': 'user', 'content': 'Now check price for nepal'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cFfxRzQywCDRu8ueqPA0VIaC', function=Function(arguments='{"destination_city":"Nepal"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'The price of a ticket to Nepal is Unknown City', 'tool_call_id': 'call_cFfxRzQywCDRu8ueqPA0VIaC'}
